In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
 %python
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "etl_vinkos")  
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsvinkosetl")



In [0]:
%python
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/*.txt"

In [0]:
from pyspark.sql.functions import *

# Lectura de todos los TXT
df_estadistica_contenedor = (
    spark.read
        .option("header", "true")
        .csv(ruta)
        .withColumn(
            "archivo_origen",
            regexp_extract(
                input_file_name(),
                r'([^/]+$)',
                1
            )
        )
        .withColumn(
            "fecha_carga",
            current_timestamp()
        )
)

# Si alguna columna no existe la creamos vacía
if "jk" not in df_estadistica_contenedor.columns:
    df_estadistica_contenedor = (
        df_estadistica_contenedor
        .withColumn("jk", lit(None).cast("string"))
    )

if "jyv" not in df_estadistica_contenedor.columns:
    df_estadistica_contenedor = (
        df_estadistica_contenedor
        .withColumn("jyv", lit(None).cast("string"))
    )

if "fgh" not in df_estadistica_contenedor.columns:
    df_estadistica_contenedor = (
        df_estadistica_contenedor
        .withColumn("fgh", lit(None).cast("string"))
    )

# Unificar jk, jyv y fgh en una sola columna llamada jyv
df_estadistica_contenedor = (
    df_estadistica_contenedor
    .withColumn(
        "jyv",
        coalesce(
            col("jk"),
            col("jyv"),
            col("fgh")
        )
    )
)

# Seleccionar únicamente las columnas del DDL Bronze
df_estad_contenedor = (

    df_estadistica_contenedor.select(

        col("email"),

        col("jyv"),

        col("Badmail").alias("badmail"),

        col("Baja").alias("baja"),

        col("Fecha envio").alias("fecha_envio"),

        col("Fecha open").alias("fecha_open"),

        col("Opens").alias("opens"),

        col("Opens virales").alias("opens_virales"),

        col("Fecha click").alias("fecha_click"),

        col("Clicks").alias("clicks"),

        col("Clicks virales").alias("clicks_virales"),

        col("Links").alias("links"),

        col("IPs").alias("ips"),

        col("Navegadores").alias("navegadores"),

        col("Plataformas").alias("plataformas"),

        col("archivo_origen"),

        col("fecha_carga")

    )

)

display(df_estad_contenedor)

#df_estad_contenedor.printSchema()

In [0]:
#Se escribe en la bitacora los nuevos valores
df_estad_contenedor.write \
    .mode("overwrite") \
    .saveAsTable(
        f"{catalogo}.{esquema}.estadistica_contenedor"
    )



In [0]:
#Se muestra lo que tiene la bitacora de control

df = spark.table(f"{catalogo}.{esquema}.estadistica_contenedor")

display(df)